# Boundary Element Methods

Boundary element methods solve boundary value problems by representing the solution with layer potentials on the boundary.

Typical model problems:

- Laplace,
- Helmholtz,
- Maxwell,
- elasticity / Lamé.

Main tradeoff:

| Pro | Contra |
| --- | --- |
| unknowns live only on the boundary | integral kernels are singular |
| exterior domains are handled naturally by the fundamental solution | discrete operators are dense |
| solution can be evaluated anywhere from the layer potential | volume source terms need additional potentials |

## Solution Representation

Model exterior Helmholtz problem:

$$
  -\Delta u - \kappa^2 u = 0 \quad \text{in } \Omega,
  \qquad
  u = g \quad \text{on } \Gamma,
$$

with the Sommerfeld radiation condition at infinity. The outgoing fundamental solution satisfies

$$
  (-\Delta_x - \kappa^2)G_\kappa(x,y) = \delta_y,
  \qquad
  G_\kappa(x,y) = \frac{e^{i\kappa |x-y|}}{4\pi |x-y|}.
$$

Green's representation formula expresses the field through boundary data only:

$$
  u(x) = \int_\Gamma
  \left(
    G_\kappa(x,y)\,\partial_{n_y}u(y)
    - \frac{\partial G_\kappa(x,y)}{\partial n_y}\,u(y)
  \right) d\sigma_y,
  \qquad x \notin \Gamma.
$$

BEM replaces the unknown boundary data by boundary densities and chooses a layer-potential ansatz.

Single-layer potential:

$$
  (V\rho)(x) = \int_\Gamma G_\kappa(x,y)\,\rho(y)\,d\sigma_y.
$$

Double-layer potential:

$$
  (K\mu)(x) = \int_\Gamma
  \frac{\partial G_\kappa(x,y)}{\partial n_y}\,\mu(y)\,d\sigma_y.
$$

For example, with a single-layer ansatz $u = V\rho$, the Dirichlet condition gives the boundary integral equation

$$
  V\rho = g \quad \text{on } \Gamma.
$$

With a double-layer ansatz, the trace jump gives an equation of the form

$$
  \left(\pm \frac12 I + K\right)\mu = g \quad \text{on } \Gamma,
$$

where the sign depends on the interior/exterior trace convention. The numerical unknown is the boundary density; after solving for it, the same potential formula evaluates $u(x)$ anywhere away from the boundary.

## Boundary Discretization

For the single-layer ansatz, the boundary integral equation is

$$
  V\rho = g \quad \text{on } \Gamma,
  \qquad
  (V\rho)(x) = \int_\Gamma G_\kappa(x,y)\,\rho(y)\,d\sigma_y.
$$

Choose a finite-dimensional boundary space

$$
  X_h = \operatorname{span}\{\varphi_1,\dots,\varphi_N\}.
$$

The Galerkin variational formulation is: find $\rho_h \in X_h$ such that

$$
  \langle V\rho_h, \eta_h \rangle_\Gamma
  = \langle g, \eta_h \rangle_\Gamma
  \qquad \forall\,\eta_h \in X_h.
$$

Expand the unknown density in the boundary basis:

$$
  \rho_h(y) = \sum_{j=1}^N c_j\,\varphi_j(y).
$$

Testing with basis functions $\eta_i$ gives the linear system

$$
  A c = b,
$$

with

$$
  A_{ij}
  = \langle V\varphi_j, \eta_i \rangle_\Gamma
  = \int_\Gamma \int_\Gamma
      \eta_i(x)\,G_\kappa(x,y)\,\varphi_j(y)
    \,d\sigma_y\,d\sigma_x,
  \qquad
  b_i = \langle g, \eta_i \rangle_\Gamma.
$$

A matrix-vector product contains a potential evaluation: density values on source quadrature points are passed through the kernel $G_\kappa(x,y)$ and accumulated at target quadrature points.

This is the computational bottleneck: far-away boundary elements still interact through the Green's function, so the discrete operator is dense.

FMM keeps near interactions direct and replaces far interactions by compressed Green's function expansions.